# TP2 — Retrieval-Augmented Generation (Part 1)
## From Documents to a Searchable Knowledge Base

**Module 2IAiasd 9.7 — Agentic AI · IMT Mines Alès**  
**Duration:** 2 h  
**Prerequisites:** TP1 completed (ReAct agent with `bind_tools`)

---

### What you will build today

By the end of this session your internship report will be **searchable by meaning**, not just by keyword.  
You will have:

1. Loaded and inspected your PDF with `PyPDFLoader`
2. Split it into chunks using two different strategies
3. Embedded every chunk with `nomic-embed-text` via Ollama
4. Stored and persisted the vectors in a **ChromaDB** collection
5. Run your first semantic retriever and compared chunking strategies

### The bigger picture

```
  Your PDF
     │
     ▼
  [Load]  ──►  pages (list of Document objects)
     │
     ▼
  [Chunk]  ──►  smaller pieces  (recursive  |  semantic)
     │
     ▼
  [Embed]  ──►  vectors  (nomic-embed-text via Ollama)
     │
     ▼
  [Index]  ──►  ChromaDB  (persisted on disk)
     │
     ▼
  [Retrieve]  ──►  top-k chunks for a query
```

Part 2 (Thursday) adds the LLM on top to answer questions and evaluates the full pipeline with RAGAS.

---

> **Your document:** Use your **2A internship report** (PDF).  
> Place it in the same folder as this notebook and update `PDF_PATH` in Section 1.  
> A fallback sample text is provided in case you do not have your PDF available today.

## Setup — Install dependencies

Run this cell once. It installs everything needed for Parts 1 and 2.

In [3]:
%pip install -q langchain langchain-ollama langchain-community langchain-chroma pypdf chromadb sentence-transformers rank-bm25 langchain-experimental langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Pull the embedding model if not already present (~270 MB, run once)
!ollama pull nomic-embed-text
print("✅ nomic-embed-text ready")

✅ nomic-embed-text ready


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling 970aa74c0a90: 100% ▕██████████████████▏ 274 MB                         
pulling c71d239df917: 100% ▕██████████████████▏  11 KB                         
pulling ce4a164fc046: 100% ▕██████████████████▏   17 B                         
pulling 31df23ea7daa: 100% ▕██████████████████▏  420 B                         
verifying sha256 digest 
writing manifest 
removing unused layers ⠋ pulling manifest 
pulling 970aa74c0a90: 100% ▕██████████████████▏ 274 MB                         
pulling c71d239df917: 100% ▕██████████████████▏  11 KB                         
pulling ce4a164fc046: 100% ▕██████████████████▏   17 B                         
pulling 31df23ea7daa: 100% ▕██████████████████▏  420 B                         
verifying sha256 digest 
writing manifest 
removing unused layers 
success 


## Imports

In [5]:
import os
import textwrap
from pathlib import Path

# LangChain — document loaders
from langchain_community.document_loaders import PyPDFLoader

# LangChain — text splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker

# LangChain — embeddings
from langchain_ollama import OllamaEmbeddings

# LangChain — vector store
from langchain_chroma import Chroma

# Utilities
import numpy as np

print("✅ Imports OK")

C:\Users\ThinkPad\AppData\Local\Temp\ipykernel_26000\1905677204.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
C:\Users\ThinkPad\AppData\Local\Temp\ipykernel_26000\1905677204.py:10: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


✅ Imports OK


---
## Section 1 — Load your PDF

### 1.1 Configuration

In [6]:
# ── CONFIGURE THIS ────────────────────────────────────────────────────────────
PDF_PATH = "rapport_stage_Aaidoun_idriss.pdf"   # ← replace with the filename of your report
CHROMA_DIR = "./chroma_db"          # ChromaDB will persist here
EMBED_MODEL = "nomic-embed-text"    # Ollama embedding model
# ─────────────────────────────────────────────────────────────────────────────

# Check the file exists — fall back to sample text if not
if Path(PDF_PATH).exists():
    print(f"✅ Found: {PDF_PATH}")
    USE_SAMPLE = False
else:
    print(f"⚠️  '{PDF_PATH}' not found — using built-in sample text instead.")
    print("   Place your PDF next to this notebook and re-run to use your own document.")
    USE_SAMPLE = True

✅ Found: rapport_stage_Aaidoun_idriss.pdf


### 1.2 Load pages with PyPDFLoader

`PyPDFLoader` reads a PDF and returns one `Document` object per page.  
Each `Document` has two attributes:
- `page_content` — the extracted text
- `metadata` — a dict with at minimum `{"source": "...", "page": N}`

In [7]:
if not USE_SAMPLE:
    loader = PyPDFLoader(PDF_PATH)
    pages = loader.load()
    print(f"📄 Loaded {len(pages)} pages from '{PDF_PATH}'")
else:
    # ── Fallback: build synthetic Document objects from plain text ────────────
    from langchain_core.documents import Document

    SAMPLE_TEXT = """
    Chapter 1 — Introduction to Machine Learning
    Machine learning is a subfield of artificial intelligence that enables computers
    to learn from data without being explicitly programmed. The main paradigms are
    supervised learning, unsupervised learning, and reinforcement learning.
    Supervised learning uses labelled examples. Unsupervised learning finds structure
    in unlabelled data. Reinforcement learning trains agents through reward signals.

    Chapter 2 — Neural Networks
    A neural network is composed of layers of interconnected nodes called neurons.
    Deep learning refers to neural networks with many hidden layers. Backpropagation
    computes gradients used to update weights during training. Activation functions
    such as ReLU and softmax introduce non-linearity into the network.

    Chapter 3 — Large Language Models
    Large Language Models (LLMs) are neural networks trained on massive text corpora
    using the Transformer architecture. They are pre-trained with a next-token
    prediction objective and then fine-tuned for specific tasks. Examples include
    GPT-4, Llama, Mistral, and Qwen. Ollama allows running LLMs locally.

    Chapter 4 — Retrieval-Augmented Generation
    RAG combines a retriever and a generator. The retriever finds relevant text
    chunks from a vector store using semantic similarity. The generator (LLM) then
    uses those chunks as context to produce a grounded answer. RAG reduces
    hallucinations by anchoring the LLM to real documents.

    Chapter 5 — Project Results
    Our internship project achieved an accuracy of 87% on the test set. We used a
    fine-tuned BERT model for classification and a gradient boosting model as
    baseline. The main challenge was class imbalance, addressed with SMOTE oversampling.
    Future work includes deploying the model as a REST API and integrating it with
    the company's existing data pipeline.
    """

    # Split sample into ~1-page chunks manually for the fallback
    chunks = [c.strip() for c in SAMPLE_TEXT.split("Chapter ") if c.strip()]
    pages = [
        Document(
            page_content="Chapter " + c,
            metadata={"source": "sample", "page": i}
        )
        for i, c in enumerate(chunks)
    ]
    print(f"📄 Using sample text — {len(pages)} synthetic pages")

📄 Loaded 42 pages from 'rapport_stage_Aaidoun_idriss.pdf'


### 1.3 Inspect the loaded pages

In [8]:
# Basic statistics
total_chars = sum(len(p.page_content) for p in pages)
avg_chars   = total_chars / len(pages) if pages else 0

print(f"Total pages   : {len(pages)}")
print(f"Total chars   : {total_chars:,}")
print(f"Avg chars/page: {avg_chars:,.0f}")
print()

# Preview the first page
print("─" * 60)
print(f"Page 0 metadata : {pages[0].metadata}")
print(f"Page 0 preview  :")
print(textwrap.fill(pages[0].page_content[:400], width=70))
print("...")

Total pages   : 42
Total chars   : 63,310
Avg chars/page: 1,507

────────────────────────────────────────────────────────────
Page 0 metadata : {'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-08-12T07:31:07+00:00', 'author': '', 'keywords': '', 'moddate': '2026-08-12T07:31:07+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'rapport_stage_Aaidoun_idriss.pdf', 'total_pages': 42, 'page': 0, 'page_label': '1'}
Page 0 preview  :
2ème Année Cycle Ingénieur RAPPORT DE STAGE PFA Titre du projet :
EMADS vers une Data Science explicable par une architecture multi-
agents hybride Conception d’un pipeline statistiquement fondé, entre
automatisation déterministe et arbitrage conditionnel par LLM Réalisé
par :Idriss Aaidoun Niveau d’étude :4ème année, filière Informatique,
spécialité Big Data & IA Établissement :École Nationale des
...

---
## Section 2 — Chunking Strategies

A page is often too long to embed meaningfully as a single unit.  
We split each page into smaller **chunks** that fit in the embedding model's context window
and that capture one coherent idea.

We will compare two strategies:

| Strategy | How it splits | When to use |
|---|---|---|
| **Recursive** | Fixed size with overlap, respects paragraph/sentence boundaries | Fast, universal, good default |
| **Semantic** | Groups sentences by embedding similarity | Slower, better topic coherence |

### 2.1 Strategy A — RecursiveCharacterTextSplitter

In [9]:
# RecursiveCharacterTextSplitter tries to split on "\n\n", then "\n", then " ",
# in that order, so it respects natural boundaries as much as possible.

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # target number of characters per chunk
    chunk_overlap=100,    # overlap between consecutive chunks (preserves context)
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks_recursive = recursive_splitter.split_documents(pages)

print(f"Recursive chunks : {len(chunks_recursive)}")
print(f"Avg chunk size   : {sum(len(c.page_content) for c in chunks_recursive) / len(chunks_recursive):,.0f} chars")
print()
print("─" * 60)
print("First 3 chunks:")
for i, chunk in enumerate(chunks_recursive[:3]):
    print(f"\n[Chunk {i}] (page={chunk.metadata.get('page','?')}, {len(chunk.page_content)} chars)")
    print(textwrap.fill(chunk.page_content[:200], width=70))

Recursive chunks : 170
Avg chunk size   : 432 chars

────────────────────────────────────────────────────────────
First 3 chunks:

[Chunk 0] (page=0, 478 chars)
2ème Année Cycle Ingénieur RAPPORT DE STAGE PFA Titre du projet :
EMADS vers une Data Science explicable par une architecture multi-
agents hybride Conception d’un pipeline statistiquement fondé, entre

[Chunk 1] (page=0, 274 chars)
Organisme d’accueil :Harmony Technology Lieu de stage :Rabat, Maroc
Période de stage :18 juin 2026 – 18 août 2026 Encadrant Académique
:Monsieur Youssef Gahi Responsable de stage :Madame Abir Bala, R&

[Chunk 2] (page=1, 427 chars)
Remerciements Je tiens à exprimer ma sincère gratitude àHarmony
Technologypour m’avoir accueilli au sein de son équipe R&D
IA/Cybersécurité et m’avoir donné l’opportunité de contribuer, durant
ces deu


### 2.2 Strategy B — SemanticChunker

`SemanticChunker` embeds each sentence and merges consecutive sentences as long as
their semantic similarity stays above a threshold.  
It uses the same embedding model you will use for indexing — so we initialise it here.

In [12]:
# Use small requests so Ollama does not receive every sentence at once.
from ollama import ResponseError
import time

class BatchedOllamaEmbeddings(OllamaEmbeddings):
    def embed_documents(self, texts):
        vectors = []
        for start in range(0, len(texts), 8):
            batch = texts[start:start + 8]
            for attempt in range(2):
                try:
                    vectors.extend(super().embed_documents(batch))
                    break
                except ResponseError as exc:
                    # A refused /tokenize connection means Ollama's runner stopped.
                    message = str(exc).lower()
                    runner_stopped = 'tokenize' in message and ('refused' in message or 'connectex' in message)
                    if not runner_stopped:
                        raise
                    if attempt == 1:
                        raise RuntimeError(
                            'Ollama embedding runner stopped. Quit and reopen Ollama, '
                            'then rerun this cell and the semantic chunking cell. '
                            f'If it persists, run: ollama pull {self.model}'
                        ) from exc
                    time.sleep(2)
        return vectors

embeddings = BatchedOllamaEmbeddings(model=EMBED_MODEL, num_thread=2)
# This makes a real request: constructing the model alone does not check Ollama.
test_vec = embeddings.embed_documents(['This is a test sentence.'])[0]
print(f'Embedding model ready: {EMBED_MODEL}')
print(f'Vector dimension: {len(test_vec)}')

Embedding model ready: nomic-embed-text
Vector dimension: 768


In [13]:
# Split where semantic distance exceeds its 85th percentile.
# Higher percentiles generally produce fewer chunks.
semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=85,
)

# Split each page independently to preserve source and page metadata.
nonempty_pages = [page for page in pages if page.page_content.strip()]
if not nonempty_pages:
    raise ValueError('No text found. Check the PDF loading cell; scanned PDFs need OCR.')

chunks_semantic = []
for index, page in enumerate(nonempty_pages, start=1):
    page_chunks = semantic_splitter.split_documents([page])
    for chunk in page_chunks:
        chunk.metadata['chunk_method'] = 'semantic'
    chunks_semantic.extend(page_chunks)
    print(f'Processed page {index}/{len(nonempty_pages)}', end='\r')
print()

avg_chunk_size = (
    sum(len(c.page_content) for c in chunks_semantic) / len(chunks_semantic)
    if chunks_semantic else 0
)
print(f'Semantic chunks  : {len(chunks_semantic)}')
print(f'Avg chunk size   : {avg_chunk_size:,.0f} chars')
print()
print('─' * 60)
print('First 3 chunks:')
for i, chunk in enumerate(chunks_semantic[:3]):
    print(f"\n[Chunk {i}] (page={chunk.metadata.get('page', '?')}, {len(chunk.page_content)} chars)")
    print(textwrap.fill(chunk.page_content[:200], width=70))

Processed page 42/42
Semantic chunks  : 249
Avg chunk size   : 253 chars

────────────────────────────────────────────────────────────
First 3 chunks:

[Chunk 0] (page=0, 713 chars)
2ème Année Cycle Ingénieur RAPPORT DE STAGE PFA Titre du projet :
EMADS vers une Data Science explicable par une architecture multi-
agents hybride Conception d’un pipeline statistiquement fondé, entre

[Chunk 1] (page=1, 818 chars)
Remerciements Je tiens à exprimer ma sincère gratitude àHarmony
Technologypour m’avoir accueilli au sein de son équipe R&D
IA/Cybersécurité et m’avoir donné l’opportunité de contribuer, durant
ces deu

[Chunk 2] (page=1, 438 chars)
J’adresse également mes remerciements àMonsieur Youssef Gahi, mon
encadrant académique à l’ENSA de Kénitra, pour le suivi apporté à ce
stage et pour ses conseils sur la structuration de ce travail. En
